# Experimentos de mejora: Katachi con imágenes DESI

Este cuaderno corre, uno tras otro y sin intervención, una lista de experimentos sobre la cadena de Katachi entrenada con FITS de DESI, y guarda los resultados de cada uno. Parte de la **corrida 2** (todas las capas entrenables), que en validación obtuvo RMSE de 0.224 dex en masa, 0.631 dex en SFR y 0.075 en D4000.

Cada experimento cambia **una sola cosa** respecto a la línea base, para saber qué efecto tiene cada mejora. Las mejoras y su justificación están en `Documentación/Propuesta_mejoras_preprocesamiento_DESI.md`; en el código son opciones de `MejorasConfig` (`src/katachi_fits/config.py`), todas apagadas por defecto.

**Reglas:**
- Todo se mide en **validación**. El grupo de prueba no se usa aquí.
- Para explorar rápido, todas las corridas usan el mismo programa corto: 30 épocas con tasa de aprendizaje en coseno y precisión mixta. Lo que importa es la diferencia contra la línea base, que usa el mismo programa. Las mejoras que ganen se confirman al final con la receta completa y tres semillas.
- Si Colab se desconecta, basta con ejecutar todo otra vez: los experimentos terminados se saltan y el que quedó a medias se reanuda.

## 1. Parámetros

In [ ]:
import os
env = os.environ.get            # las variables EXP_* solo se usan en la prueba local reducida

EPOCAS = int(env("EXP_EPOCAS", "30"))         # épocas del programa rápido
SEMILLA = 42
USAR_WANDB = env("EXP_WANDB", "0") == "1"     # True para ver las curvas en vivo en wandb.ai
PROYECTO_WANDB = "katachi-desi"
LIMITE = int(env("EXP_LIMIT", "0"))           # 0 = todas las galaxias (prueba local: pocas)
SOLO = env("EXP_SOLO", "")                    # nombres separados por comas para correr solo algunos

## 2. Entorno

In [ ]:
import sys, json, time, subprocess, platform, warnings
from pathlib import Path

EN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/erteck/InferenciaPropiedadesFisicas.git"
REPO_DIR = Path(env("EXP_REPO_DIR", "/content/InferenciaPropiedadesFisicas"))
if EN_COLAB:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "hickle"] + (["wandb"] if USAR_WANDB else []), check=True)
    from google.colab import drive
    drive.mount("/content/drive")
    if not REPO_DIR.exists():
        subprocess.run(["git", "clone", "-q", REPO_URL, str(REPO_DIR)], check=True)
    subprocess.run(["git", "-C", str(REPO_DIR), "pull", "-q"], check=True)
sys.path.insert(0, str(REPO_DIR))
COMMIT = subprocess.run(["git", "-C", str(REPO_DIR), "rev-parse", "HEAD"], capture_output=True, text=True).stdout.strip()

warnings.filterwarnings("ignore")
import numpy as np, pandas as pd, torch
import matplotlib.pyplot as plt
from IPython.display import display

from src.katachi_fits import config as K, catalog as C
from src.katachi_fits.cache import build_cache, cache_dir, load_cache
from src.katachi_fits.config import FitsConfig, MejorasConfig, Paths, TrainConfig
from src.katachi_fits.experimentos import Experimento, correr_lista, preparar_datos
from src.katachi_fits.io_guard import WriteGuard
from src.katachi_fits.transforms import Normalizer
from src.katachi_fits.variants import DESI

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
pd.set_option("display.precision", 4, "display.max_columns", 60, "display.width", 250)
print("commit", COMMIT[:12], "·", DEVICE, torch.cuda.get_device_name(0) if DEVICE.type == "cuda" else "")
if USAR_WANDB:
    import wandb
    wandb.login()            # pide la clave la primera vez: https://wandb.ai/authorize

## 3. Datos

Usamos la misma muestra, caché y normalización que la corrida 2, leídas de Drive. La primera vez que algún experimento pide el **margen de rotación**, se construye además una caché de recortes de 440 px (unos 20 minutos); después se reutiliza.

In [ ]:
BASE = Path(env("EXP_BASE", str(K.DRIVE_BASE)))
DESCARGA = BASE / K.DOWNLOAD_DIRNAME
PLAN_CSV = next(DESCARGA.glob("**/plan.csv"))
CATALOGO = next(p for n in K.TEAM_CATALOG_NAMES for p in DESCARGA.glob(f"**/{n}"))
paths = Paths(base=BASE, plan_csv=PLAN_CSV, inventario_csv=PLAN_CSV.parent / "inventario.csv",
              fits_dir=Path(env("EXP_FITS_DIR", str(DESCARGA / "fits"))),
              results=Path(env("EXP_RESULTS", str(BASE / K.RESULTS_DIRNAME))), team_catalog=CATALOGO)
guard = WriteGuard(paths.results, paths.read_only)

plan = C.load_plan(PLAN_CSV, paths.fits_dir)
equipo = C.load_team_catalog(CATALOGO)
scalars = C.load_scalars(paths.external / "scalars.cat")
cohorte, _ = C.build_cohort_team(plan, equipo, scalars)
if LIMITE:
    cohorte = (cohorte.groupby("particion", group_keys=False)
                      .apply(lambda d: d.head(max(LIMITE // 4, 8) if d.name != "Train" else LIMITE)).reset_index(drop=True))
cohorte, _ = C.apply_qc(cohorte, K.QC_GROUPS_INCLUDED)
TAG = f"_lim{LIMITE}" if LIMITE else ""
print("Muestra:", cohorte.particion.value_counts().to_dict())

fcfg = FitsConfig()
CACHE = cache_dir(paths.cache, DESI, fcfg, TAG)
build_cache(cohorte, "ruta_local", CACHE, guard, fcfg, variant=DESI, chunk=500, workers=24)
X300, ids, _, _ = load_cache(CACHE, dtype=fcfg.storage_dtype)
assert list(ids) == list(cohorte.mangaid)
NORM = Normalizer.from_json((paths.results / "normalizacion" / f"normalizador_recorte300_beta3{TAG}.json").read_text())
print("Caché:", X300.shape, "· σ:", np.round(NORM.sigma, 5), "· s_b:", np.round(NORM.scale, 3))

## 4. Lista de experimentos

Todos comparten el programa rápido y la precisión mixta. El primero es la **línea base** del lote: la receta de la corrida 2 con ese mismo programa. Cada uno de los demás enciende una mejora:

| Experimento | Qué cambia | Problema que ataca |
|---|---|---|
| `base` | Nada | Referencia del lote |
| `margen` | Caché de 440 px: se gira y después se recorta, sin esquinas vacías; interpolación bilineal | Con 300 px, al girar quedan esquinas en 0 sin ruido (hasta 17 % de la imagen), que no aparecen en validación |
| `ruido_psf` | `margen` + ruido con el nivel del cielo, desenfoque leve y desplazamiento de ±3 px | La red era sensible al ruido; los telescopios del norte y del sur tienen distinta calidad |
| `planos` | Planos de masa y SFR estandarizados | El plano de masa está en una escala 80 veces mayor que la imagen |
| `std_imagen` | Imagen con media 0 y desviación 1 por banda | Los pesos de ImageNet esperan esa escala |
| `sigma_img` | σ del cielo de cada imagen, no uno global | La profundidad cambia entre imágenes y entre regiones |
| `todo` | Todas las anteriores juntas | Saber si los efectos se suman |

Además, cada experimento se evalúa **con y sin TTA** (promedio de las 8 versiones giradas y reflejadas de cada galaxia), sin entrenar de nuevo.

Para agregar un experimento basta con añadir una línea a la lista.

In [ ]:
RAPIDO = dict(programa_rapido=True, epocas_rapido=EPOCAS, precision_mixta=True)
GEOM = dict(margen_rotacion=True, rotacion_bilineal=True)
EXPERIMENTOS = [
    Experimento("base", MejorasConfig(**RAPIDO), SEMILLA, descripcion="Corrida 2 con el programa rápido"),
    Experimento("margen", MejorasConfig(**RAPIDO, **GEOM), SEMILLA, descripcion="Rotación sin esquinas vacías"),
    Experimento("ruido_psf", MejorasConfig(**RAPIDO, **GEOM, ruido_max=1.0, psf_max_px=1.0, desplazamiento_px=3),
                SEMILLA, descripcion="+ ruido, desenfoque y desplazamiento"),
    Experimento("planos", MejorasConfig(**RAPIDO, estandarizar_planos=True), SEMILLA, descripcion="Planos estandarizados"),
    Experimento("std_imagen", MejorasConfig(**RAPIDO, estandarizar_imagen=True), SEMILLA, descripcion="Imagen estandarizada"),
    Experimento("sigma_img", MejorasConfig(**RAPIDO, sigma_por_imagen=True), SEMILLA, descripcion="σ del cielo por imagen"),
    Experimento("todo", MejorasConfig(**RAPIDO, **GEOM, ruido_max=1.0, psf_max_px=1.0, desplazamiento_px=3,
                                      estandarizar_planos=True, estandarizar_imagen=True, sigma_por_imagen=True),
                SEMILLA, descripcion="Todas las mejoras"),
]
if SOLO:
    EXPERIMENTOS = [e for e in EXPERIMENTOS if e.nombre == "base" or e.nombre in SOLO.split(",")]
display(pd.DataFrame([{"experimento": e.nombre, "descripción": e.descripcion, "identificador": e.identificador(300)}
                      for e in EXPERIMENTOS]))

## 5. Correr la lista

La celda siguiente corre todos los experimentos en orden. Por cada uno guarda en `resultados_katachi_fits/runs/<identificador>/`:
- su configuración (`config.json`);
- el historial por época (`history.csv`);
- los pesos (`final_model.pt`);
- las predicciones de validación con y sin TTA;
- sus métricas (`resultado.json`).

La tabla de todos los experimentos se va actualizando en `resultados_katachi_fits/experimentos/resultados_experimentos.csv`.

En una A100, cada experimento tarda unos 10–15 minutos con 30 épocas.

In [ ]:
CTX = preparar_datos(cohorte, X300, NORM, fcfg, paths, guard, EXPERIMENTOS, tag=TAG)
t0 = time.time()
TABLA = correr_lista(EXPERIMENTOS, CTX, paths, guard, DEVICE,
                     base=TrainConfig(fused_adam=DEVICE.type == "cuda", channels_last=DEVICE.type == "cuda"),
                     # Solo la prueba local reducida limita las épocas; en Colab la receta completa
                     # de las corridas de confirmación corre hasta su parada natural.
                     max_epochs=EPOCAS if LIMITE else None,
                     wandb_proyecto=PROYECTO_WANDB if USAR_WANDB else None,
                     log=lambda s: print(s, flush=True))
print(f"\nLista terminada en {(time.time() - t0) / 60:.0f} min")

## 6. Resultados

La primera tabla muestra el RMSE de cada experimento en validación, con y sin TTA. La segunda compara cada experimento con la línea base sobre las mismas galaxias: ΔRMSE es la diferencia (negativo = mejor), con su intervalo de confianza del 95 % por *bootstrap* pareado, y el veredicto sigue la misma regla de decisión de los cuadernos anteriores:

| Veredicto | Significado |
|---|---|
| **mejor** | Todo el intervalo de ΔRMSE está por debajo de 0 |
| **peor** | Todo el intervalo está por encima de 0 |
| **equivalente** | El intervalo cae dentro de ±0.02 dex en masa y SFR, o ±0.01 en D4000 |
| **no concluyente** | Ninguno de los anteriores |

In [ ]:
cols = ["experimento", "epocas", "s_por_epoca", "min_entrenamiento"]
for n in ("M*", "SFR", "D4000"):
    cols += [f"RMSE_{n}", f"IC_{n}", f"RMSE_{n}_tta"]
display(TABLA[cols].set_index("experimento"))

comp = TABLA.set_index("experimento")[[c for c in TABLA.columns if c.startswith(("dRMSE", "veredicto"))]]
display(comp.iloc[1:])

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(15, 4.2), sharey=True)
yy = np.arange(len(TABLA))
for a, n in zip(ax, ("M*", "SFR", "D4000")):
    ic = np.array([[float(v) for v in s.strip("[]").split(",")] for s in TABLA[f"IC_{n}"]])
    r = TABLA[f"RMSE_{n}"].to_numpy()
    a.errorbar(r, yy, xerr=np.abs(ic.T - r), fmt="o", capsize=3, label="sin TTA")
    a.plot(TABLA[f"RMSE_{n}_tta"], yy + 0.2, "s", label="con TTA")
    a.axvline(r[0], color="gray", ls=":", label="línea base del lote")
    a.set(xlabel=f"RMSE en validación · {n}")
ax[0].set_yticks(yy); ax[0].set_yticklabels(TABLA.experimento); ax[0].invert_yaxis(); ax[0].legend(fontsize=8)
plt.tight_layout(); plt.show()

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(15, 3.8))
for _, f in TABLA.iterrows():
    h = pd.read_csv(paths.run_dir(f["id"]) / "history.csv")
    for a, k in zip(ax, ("mstar", "sfr", "d4000")):
        a.plot(h.epoch, h[f"val_loss_{k}"], label=f.experimento)
for a, t in zip(ax, ("log M*", "log SFR", "D4000")):
    a.set(yscale="log", xlabel="época", ylabel="MSE en validación", title=t)
ax[0].legend(fontsize=7); plt.tight_layout(); plt.show()

## 7. Cómo leer los resultados y qué sigue

1. **Aceptar una mejora** solo si su veredicto es *mejor* en masa o en SFR y ninguna otra propiedad sale *peor*.
2. **TTA:** si la columna `_tta` es consistentemente menor, conviene usarla en la evaluación final; no requiere reentrenar.
3. **Combinación:** si `todo` no supera a la mejor mejora individual, alguna de ellas estorba; conviene combinar solo las aceptadas.
4. **Confirmación:** con la mejor configuración, correr la receta completa de Katachi (`programa_rapido=False`) con las semillas 42, 7 y 13. Basta con agregarlas a la lista:

```python
mejor = MejorasConfig(...)          # las opciones aceptadas, sin programa_rapido
EXPERIMENTOS = [Experimento("final", mejor, s, descripcion="Confirmación") for s in (42, 7, 13)]
```

5. **Grupo de prueba:** solo después de la confirmación, una sola vez, con el cuaderno `Katachi_DESI_FITS.ipynb`.

Hay que cuidar un riesgo: con muchas comparaciones en validación, alguna puede salir *mejor* por azar. Por eso la confirmación con tres semillas es obligatoria antes de abrir el grupo de prueba.